# 14 Full Video Baseline

Consumes contact-aligned `clips_v1` rows and writes video baseline embeddings, predictions, and metrics. The default encoder is a local-safe OpenCV feature extractor so the pipeline can be tested before large model downloads.

Required inputs:

- `manifests/bbe_events_v1.parquet`
- `clips/{FULL_RUN_ID}/clips_v1.parquet` with real `clip_path` values
- `configs/targets/target_registry_v1.yaml`

Created outputs:

- `features/{video_lightweight_feature_id}/manifest.parquet`
- `predictions/video_lightweight_cv2_mlb_2024_2026_v1/predictions_v1.parquet`
- `predictions/video_lightweight_cv2_mlb_2024_2026_v1/metrics_v1.json`

Next: `notebooks/15_full_fusion.ipynb`.

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
VIDEO_RUN_ID = run_id(RUN_PROFILE, 'video_lightweight_run_id', run_id(RUN_PROFILE, 'video_run_id'))
VIDEO_LIGHTWEIGHT_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_lightweight_feature_id', 'video_lightweight_features_mlb_2024_2026_v2')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('VIDEO_RUN_ID =', VIDEO_RUN_ID)
print('VIDEO_LIGHTWEIGHT_FEATURE_ID =', VIDEO_LIGHTWEIGHT_FEATURE_ID)


## GPU / Input Check

The default lightweight encoder can run on CPU, but real frozen VideoMAE embedding extraction should use L4 or A100 and should be enabled explicitly in a later heavy-model cell.

In [ ]:
from sport_pipeline.artifact_check import check_artifacts
from sport_pipeline.runtime.device import summarize_runtime_device

print(summarize_runtime_device(prefer_gpu=True, require_gpu=False))
required = [
    'manifests/bbe_events_v1.parquet',
    f'clips/{FULL_RUN_ID}/clips_v1.parquet',
]
print(check_artifacts(BASE_DIR, required))

## Run Video Baseline

`MAX_CLIPS=None` processes every representative event clip with a valid `clip_path`. Set a small integer for a pilot.

In [ ]:
from sport_pipeline.models.video.full_baseline import run_full_video_baseline

VIDEO_SETTINGS = stage_settings(RUN_PROFILE, 'lightweight_video')
MAX_CLIPS = VIDEO_SETTINGS.get('max_clips')
MAX_FRAMES = int(VIDEO_SETTINGS.get('max_frames', 16))
CACHE_INPUTS = bool(VIDEO_SETTINGS.get('cache_inputs', False))
CACHE_NUM_WORKERS = int(VIDEO_SETTINGS.get('cache_num_workers', 4))
CACHE_MIN_FREE_DISK_GB = float(VIDEO_SETTINGS.get('cache_min_free_disk_gb', 20))
CACHE_MAX_FILE_MB = VIDEO_SETTINGS.get('cache_max_file_mb')

print('lightweight_video cache_inputs =', CACHE_INPUTS, 'cache_num_workers =', CACHE_NUM_WORKERS)

outputs = run_full_video_baseline(
    BASE_DIR,
    clip_run_id=FULL_RUN_ID,
    prediction_run_id=VIDEO_RUN_ID,
    max_clips=MAX_CLIPS,
    max_frames=MAX_FRAMES,
    encoder_mode='lightweight',
    require_non_empty=True,
    video_embedding_feature_id=VIDEO_LIGHTWEIGHT_FEATURE_ID,
    cache_dir=CACHE_DIR,
    cache_inputs=CACHE_INPUTS,
    cache_num_workers=CACHE_NUM_WORKERS,
    cache_min_free_disk_gb=CACHE_MIN_FREE_DISK_GB,
    cache_max_file_mb=CACHE_MAX_FILE_MB,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


## Output Check

In [ ]:
expected = [
    f'features/{VIDEO_LIGHTWEIGHT_FEATURE_ID}/manifest.parquet',
    f'predictions/{VIDEO_RUN_ID}/predictions_v1.parquet',
    f'predictions/{VIDEO_RUN_ID}/metrics_v1.json',
    f'reports/preflight/full_video_baseline_{VIDEO_RUN_ID}.json',
]
result = check_artifacts(BASE_DIR, expected)
print(result)
print('NEXT: notebooks/15_full_fusion.ipynb')